# CheXzero Baseline — Google Colab

**Project:** Vision-Language Models in Radiology  
**Repo:** https://github.com/SinaDns/radiology-vision-language-models

This notebook:
1. Clones the repository and installs dependencies
2. Downloads the IU X-Ray dataset
3. Runs import / forward-pass smoke tests
4. Trains CheXzero contrastively on IU X-Ray
5. Evaluates zero-shot classification on NIH ChestX-ray14

> **Runtime:** Set *Runtime → Change runtime type → T4 GPU* before running.
>
> **NIH-14 evaluation (Section 5) requires a manual download step** — see that section for instructions.

---
## 1. Environment Setup

In [ ]:
import os, subprocess, sys

# ── Clone / pull repo ────────────────────────────────────────────────────────
REPO_URL  = "https://github.com/SinaDns/radiology-vision-language-models.git"
REPO_DIR  = "/content/radiology-vision-language-models"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
# ── Install Python dependencies ──────────────────────────────────────────────
# Colab already ships with torch/torchvision; we upgrade them if needed
# and install the remaining packages from requirements.txt.
!pip install -q -r requirements.txt

In [1]:
# ── Logging + GPU check ───────────────────────────────────────────────────────
import logging, torch

# Configure root logger so all src.* module logs appear in Colab output.
# Each log line shows timestamp, level, and which module emitted the message,
# which makes remote debugging from logs much easier.
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s — %(message)s',
    datefmt='%H:%M:%S',
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


---
## 2. Download IU X-Ray Dataset

~7,470 PNG images + ~3,955 XML reports from the NIH OpenI collection.  
Total size ≈ 1.4 GB.

In [4]:
import os
from pathlib import Path

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd().resolve().parent
    
DATA_DIR = BASE_DIR / "data" / "iu_xray"
DATA_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_TGZ  = DATA_DIR / "NLMCXR_png.tgz"
REPORT_TGZ = DATA_DIR / "NLMCXR_reports.tgz"

# Download (skip if already present)
if not IMAGE_TGZ.exists():
    print("Downloading images (~1.3 GB)…")
    !wget -q --show-progress \
        "https://openi.nlm.nih.gov/imgs/collections/NLMCXR_png.tgz" \
        -O {IMAGE_TGZ}
else:
    print("Image archive already present.")

if not REPORT_TGZ.exists():
    print("Downloading reports…")
    !wget -q --show-progress \
        "https://openi.nlm.nih.gov/imgs/collections/NLMCXR_reports.tgz" \
        -O {REPORT_TGZ}
else:
    print("Report archive already present.")

/home/dev/KD-projec 100%[===================>]   1.27G   349KB/s    in 1h 47m  
/home/dev/KD-projec 100%[===================>]   1.06M   352KB/s    in 3.1s    


In [12]:
import sys, os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [6]:
# ── Extract ──────────────────────────────────────────────────────────────────
IMAGES_DIR  = DATA_DIR / "images"
REPORTS_DIR = DATA_DIR / "reports"
IMAGES_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

import tarfile

if not any(IMAGES_DIR.glob("*.png")):
    print("Extracting images…")
    with tarfile.open(IMAGE_TGZ, "r:gz") as tar:
        for member in tar.getmembers():
            member.name = Path(member.name).name  # flatten directory
            tar.extract(member, IMAGES_DIR)
    print("Done.")

if not any(REPORTS_DIR.glob("*.xml")):
    print("Extracting reports…")
    with tarfile.open(REPORT_TGZ, "r:gz") as tar:
        for member in tar.getmembers():
            member.name = Path(member.name).name
            tar.extract(member, REPORTS_DIR)
    print("Done.")

n_images  = len(list(IMAGES_DIR.glob("*.png")))
n_reports = len(list(REPORTS_DIR.glob("*.xml")))
print(f"PNG images : {n_images}   (expected ~7,470)")
print(f"XML reports: {n_reports}  (expected ~3,955)")

Extracting images…
Done.
Extracting reports…
Done.
PNG images : 7470   (expected ~7,470)
XML reports: 3955  (expected ~3,955)


---
## 3. Smoke Tests (no data required)

In [13]:
REPO_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() \
            else Path(os.getcwd()).parent if Path(os.getcwd()).name == "experiments" \
            else Path(os.getcwd())

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repo root :", REPO_ROOT)
print("CWD       :", os.getcwd())
print("src exists:", (REPO_ROOT / "src").is_dir())

Repo root : /home/dev/KD-project/radiology-vision-language-models
CWD       : /home/dev/KD-project/radiology-vision-language-models
src exists: True


In [15]:
# ── Data loader import ───────────────────────────────────────────────────────
from src.data_loaders.iu_xray import IUXrayDataset
from src.data_loaders.nih_chestxray14 import NIHChestXray14Dataset
from src.data_loaders.transforms import get_train_transforms, get_val_transforms
print("Data loaders OK")

Data loaders OK


In [16]:
# ── Model forward pass ───────────────────────────────────────────────────────
import torch
from src.models.chexzero import CheXzero

# This downloads BioClinicalBERT weights (~400 MB) on the first run.
model = CheXzero(embed_dim=512)

imgs  = torch.randn(2, 3, 320, 320)
ids   = torch.randint(0, 1000, (2, 64))
mask  = torch.ones(2, 64, dtype=torch.long)

logits_i, logits_t = model(imgs, ids, mask)
print("Forward pass OK — logits shape:", logits_i.shape)  # expected (2, 2)

/home/dev/KD-project/radiology-vision-language-models/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /home/dev/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:33<00:00, 3.07MB/s]
09:08:52 [INFO] src.models.chexzero — ImageEncoder — embed_dim=512 pretrained=True params=24.56M
09:08:52 [INFO] src.models.chexzero — TextEncoder — loading emilyalsentzer/Bio_ClinicalBERT …
09:08:53 [INFO] httpx — HTTP Request: HEAD https://huggingface.co/emilyalsentzer/Bio_ClinicalBERT/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
09:08:53 [INFO] httpx — HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/emilyalsentzer/Bio_ClinicalBERT/d5892b39a4adaed74b92212a44081509db72f87b/config.json "HTTP/1.1 200 OK"
09:08:54 [INFO] httpx — HTTP Request: GET https://huggingface.co/api/resolve-cache/models/emilyalsentzer/Bio_ClinicalBERT/d5892b39a4adaed74b92212a44081509db72f87b/config.json "HTTP/1.1 200 OK"
09:08:55 [INFO] httpx — HTTP Request: HEAD https://huggingface.co/emilyalsentzer/Bio_ClinicalBERT/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
09:08:55 [INFO] httpx — HTTP Request: HEAD https://hugging

Forward pass OK — logits shape: torch.Size([2, 2])


In [17]:
# ── Config loading ───────────────────────────────────────────────────────────
from src.utils.config import load_config
config = load_config("experiments/configs/chexzero.yaml")
print(config)

{'model': {'embed_dim': 512, 'temperature': 0.07}, 'training': {'batch_size': 128, 'lr': 5e-05, 'weight_decay': 0.0001, 'epochs': 20, 'grad_clip': 1.0, 'mixed_precision': True}, 'data': {'image_size': 320, 'num_workers': 4, 'val_split': 0.1}, 'paths': {'iu_xray_dir': 'data/iu_xray/', 'nih14_dir': 'data/nih_chestxray14/', 'checkpoint_dir': 'experiments/results/checkpoints/chexzero/', 'log_dir': 'experiments/results/logs/'}, 'logging': {'use_wandb': False, 'project': 'radiology-vlm', 'run_name': 'chexzero-baseline'}}


'The read operation timed out' thrown while requesting GET https://huggingface.co/api/models/emilyalsentzer/Bio_ClinicalBERT/xet-read-token/3c22c28ae9c1619228e31dc7630645fee6081c98
09:09:38 [WARNING] huggingface_hub.utils._http — 'The read operation timed out' thrown while requesting GET https://huggingface.co/api/models/emilyalsentzer/Bio_ClinicalBERT/xet-read-token/3c22c28ae9c1619228e31dc7630645fee6081c98
Retrying in 1s [Retry 1/5].
09:09:38 [WARNING] huggingface_hub.utils._http — Retrying in 1s [Retry 1/5].
09:09:44 [INFO] httpx — HTTP Request: GET https://huggingface.co/api/models/emilyalsentzer/Bio_ClinicalBERT/xet-read-token/3c22c28ae9c1619228e31dc7630645fee6081c98 "HTTP/1.1 200 OK"


In [18]:
# ── Loss function sanity check ───────────────────────────────────────────────
from src.training.losses import clip_loss
B = 4
logits = torch.eye(B) * 10  # perfect alignment
loss = clip_loss(logits, logits.T)
print(f"clip_loss (perfect alignment, B={B}): {loss.item():.4f}  (should be near 0)")

clip_loss (perfect alignment, B=4): 0.0001  (should be near 0)


---
## 4. Training

Trains CheXzero contrastively on IU X-Ray.  
Checkpoints are saved to `experiments/results/checkpoints/chexzero/`.

In [19]:
# ── Config overrides for Colab ───────────────────────────────────────────────
# Colab T4 has 15 GB VRAM. batch_size=32 is safe; increase to 64 for A100.

from src.utils.config import load_config
import copy

config = load_config("experiments/configs/chexzero.yaml")

# Point to Colab data paths
config["paths"]["iu_xray_dir"]     = "data/iu_xray/"
config["paths"]["nih14_dir"]       = "data/nih_chestxray14/"
config["paths"]["checkpoint_dir"]  = "experiments/results/checkpoints/chexzero/"
config["paths"]["log_dir"]         = "experiments/results/logs/"

# T4-safe: BS=32 (ResNet-50 + BioClinicalBERT + 320×320 images)
# Increase to 64 for A100 / 128 for multi-GPU
config["training"]["batch_size"] = 32

# ~30 min on T4; increase to 20 for full training
config["training"]["epochs"] = 10

# Reduce workers to avoid Colab shared-memory issues
config["data"]["num_workers"] = 2

print("Config ready:", config)

Config ready: {'model': {'embed_dim': 512, 'temperature': 0.07}, 'training': {'batch_size': 32, 'lr': 5e-05, 'weight_decay': 0.0001, 'epochs': 10, 'grad_clip': 1.0, 'mixed_precision': True}, 'data': {'image_size': 320, 'num_workers': 2, 'val_split': 0.1}, 'paths': {'iu_xray_dir': 'data/iu_xray/', 'nih14_dir': 'data/nih_chestxray14/', 'checkpoint_dir': 'experiments/results/checkpoints/chexzero/', 'log_dir': 'experiments/results/logs/'}, 'logging': {'use_wandb': False, 'project': 'radiology-vlm', 'run_name': 'chexzero-baseline'}}


In [23]:
try:
    BASE_DIR = Path(__file__).resolve()
except NameError:
    BASE_DIR = Path.cwd().resolve()
    
DATA_DIR = BASE_DIR / "data" / "iu_xray"

In [ ]:
import torch
from torch.utils.data import DataLoader

from src.data_loaders.iu_xray import IUXrayDataset
from src.data_loaders.transforms import get_train_transforms, get_val_transforms
from src.models.chexzero import CheXzero
from src.training.contrastive import ContrastiveTrainer
from src.utils.logging_utils import setup_logger

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

image_size   = config["data"]["image_size"]
val_fraction = config["data"]["val_split"]
iu_xray_dir  = config["paths"]["iu_xray_dir"]

train_dataset = IUXrayDataset(
    data_dir=iu_xray_dir,
    split="train",
    val_fraction=val_fraction,
    transform=get_train_transforms(image_size),
)
val_dataset = IUXrayDataset(
    data_dir=iu_xray_dir,
    split="val",
    val_fraction=val_fraction,
    transform=get_val_transforms(image_size),
)
print(f"Train: {len(train_dataset)} samples  |  Val: {len(val_dataset)} samples")

batch_size   = config["training"]["batch_size"]
num_workers  = config["data"]["num_workers"]

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
    drop_last=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
)

model  = CheXzero(embed_dim=config["model"]["embed_dim"])
logger = setup_logger(config["paths"]["log_dir"], config["logging"]["run_name"])

trainer = ContrastiveTrainer(
    model=model,
    config=config,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    logger=logger,
)

print("Trainer ready.")

09:13:41 [INFO] src.data_loaders.iu_xray — IUXrayDataset init — data_dir=/home/dev/KD-project/radiology-vision-language-models/data/iu_xray split=train
09:13:41 [INFO] src.data_loaders.iu_xray — Found 3955 XML report files, 7470 PNG images in index


Device: cuda


09:13:42 [INFO] src.data_loaders.iu_xray — Build complete — pairs=0  skipped(empty_report)=28  skipped(no_uri)=3927  unresolved_uris=0
09:13:42 [INFO] src.data_loaders.iu_xray — Raw samples (before split): 0
09:13:42 [INFO] src.data_loaders.iu_xray — Studies total=0  val=0  train=0
09:13:42 [INFO] src.data_loaders.iu_xray — [train split] 0 image-report pairs from 0 studies
09:13:42 [WARNING] src.data_loaders.iu_xray — Dataset is EMPTY for split='train'. Verify that images/ and reports/ are populated.
09:13:42 [INFO] src.data_loaders.iu_xray — IUXrayDataset init — data_dir=/home/dev/KD-project/radiology-vision-language-models/data/iu_xray split=val
09:13:42 [INFO] src.data_loaders.iu_xray — Found 3955 XML report files, 7470 PNG images in index
09:13:43 [INFO] src.data_loaders.iu_xray — Build complete — pairs=0  skipped(empty_report)=28  skipped(no_uri)=3927  unresolved_uris=0
09:13:43 [INFO] src.data_loaders.iu_xray — Raw samples (before split): 0
09:13:43 [INFO] src.data_loaders.iu_xra

Train: 0 samples  |  Val: 0 samples


ValueError: num_samples should be a positive integer value, but got num_samples=0

In [ ]:
# ── Run training ─────────────────────────────────────────────────────────────
# Set resume_path to continue from a previous checkpoint, e.g.:
#   resume_path = "experiments/results/checkpoints/chexzero/latest.pt"

trainer.train(resume_path=None)

In [ ]:
# ── (Optional) Save checkpoint to Google Drive ───────────────────────────────
# Uncomment to persist the best checkpoint across Colab sessions.

# from google.colab import drive
# drive.mount("/content/drive")
# import shutil
# shutil.copy(
#     "experiments/results/checkpoints/chexzero/best.pt",
#     "/content/drive/MyDrive/chexzero_best.pt",
# )
# print("Checkpoint saved to Google Drive.")

---
## 4.5  IU X-Ray Quick Eval (no NIH-14 required)

Cross-modal retrieval on the **IU X-Ray validation set**: for each image, rank all
validation reports by cosine similarity and report top-1 / top-5 retrieval accuracy
(correct report = the one paired with that image).

This is a meaningful sanity check that runs immediately after training — a random
baseline scores ~1/N for top-1 and ~5/N for top-5, where N is the val set size.

In [ ]:
# ── Image→Text retrieval on IU X-Ray val set ─────────────────────────────────
import torch
import numpy as np
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from src.models.chexzero import CheXzero
from src.data_loaders.iu_xray import IUXrayDataset
from src.data_loaders.transforms import get_val_transforms
from src.utils.config import load_config

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg    = load_config("experiments/configs/chexzero.yaml")

# Load the best checkpoint
CHECKPOINT = "experiments/results/checkpoints/chexzero/best.pt"
czero = CheXzero(embed_dim=cfg["model"]["embed_dim"])
ckpt  = torch.load(CHECKPOINT, map_location=device)
czero.load_state_dict(ckpt["model_state_dict"])
czero.to(device).eval()
print(f"CheXzero loaded (epoch {ckpt.get('epoch','?')})")

# Val dataset: load all samples (no shuffle so indices are stable)
val_ds = IUXrayDataset(
    data_dir="data/iu_xray/",
    split="val",
    val_fraction=cfg["data"]["val_split"],
    transform=get_val_transforms(cfg["data"]["image_size"]),
)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False,
                        num_workers=2, pin_memory=True)
N = len(val_ds)
print(f"Val set: {N} image-report pairs")

# ── Encode all images ─────────────────────────────────────────────────────────
img_embs = []
reports_all = []

with torch.no_grad():
    for batch in val_loader:
        imgs = batch["image"].to(device)
        emb  = czero.encode_image(imgs)          # (B, 512), already L2-normed
        img_embs.append(emb.cpu())
        reports_all.extend(batch["report"])

img_embs = torch.cat(img_embs, 0)               # (N, 512)
print(f"Image embeddings: {img_embs.shape}")

# ── Encode all unique reports ─────────────────────────────────────────────────
tokenizer_bert = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
unique_reports = list(dict.fromkeys(reports_all))   # deduplicated, order preserved
report_to_idx  = {r: i for i, r in enumerate(unique_reports)}

txt_embs = []
with torch.no_grad():
    for i in range(0, len(unique_reports), 32):
        batch_texts = unique_reports[i:i+32]
        tok = tokenizer_bert(
            batch_texts, max_length=256, padding=True,
            truncation=True, return_tensors="pt",
        )
        ids  = tok["input_ids"].to(device)
        mask = tok["attention_mask"].to(device)
        emb  = czero.encode_text(ids, mask)
        txt_embs.append(emb.cpu())

txt_embs = torch.cat(txt_embs, 0)               # (M, 512)
print(f"Report embeddings: {txt_embs.shape}  (unique reports: {len(unique_reports)})")

In [ ]:
# ── Compute retrieval accuracy ────────────────────────────────────────────────
import os
import matplotlib.pyplot as plt

os.makedirs("experiments/results", exist_ok=True)

# Ground-truth: each image should retrieve its own report
# Build GT index: for image i, its correct report is reports_all[i]
gt_report_idx = np.array([report_to_idx[r] for r in reports_all])  # (N,)

# Similarity matrix: (N images) x (M unique reports)
# img_embs: (N, 512)  txt_embs: (M, 512)  — both L2-normed, so dot product = cosine sim
sim_matrix = img_embs @ txt_embs.T      # (N, M)

# Top-k retrieval accuracy
def topk_accuracy(sim, gt_idx, k):
    """Fraction of images whose gt report is in the top-k retrieved reports."""
    topk_indices = torch.topk(sim, k=k, dim=1).indices  # (N, k)
    gt_expanded  = torch.tensor(gt_idx).unsqueeze(1)     # (N, 1)
    hits         = (topk_indices == gt_expanded).any(dim=1)
    return hits.float().mean().item()

R1  = topk_accuracy(sim_matrix, gt_report_idx, k=1)
R5  = topk_accuracy(sim_matrix, gt_report_idx, k=5)
R10 = topk_accuracy(sim_matrix, gt_report_idx, k=10)

# Random baseline
M = len(unique_reports)
rand_R1  = 1.0 / M
rand_R5  = 5.0 / M
rand_R10 = 10.0 / M

print("=== IU X-Ray Image→Text Retrieval (val set) ===")
print(f"{'Metric':<12} {'Trained':>10}  {'Random':>10}")
print("-" * 36)
print(f"{'R@1':<12} {R1:>10.4f}  {rand_R1:>10.4f}")
print(f"{'R@5':<12} {R5:>10.4f}  {rand_R5:>10.4f}")
print(f"{'R@10':<12} {R10:>10.4f}  {rand_R10:>10.4f}")
print(f"\nN={N} images  M={M} unique reports")

# Bar chart
fig, ax = plt.subplots(figsize=(7, 4))
ks      = ["R@1", "R@5", "R@10"]
trained = [R1, R5, R10]
random  = [rand_R1, rand_R5, rand_R10]
x = np.arange(len(ks))
w = 0.35
ax.bar(x - w/2, trained, w, label="CheXzero", color="steelblue")
ax.bar(x + w/2, random,  w, label="Random",   color="lightgray", edgecolor="gray")
ax.set_xticks(x); ax.set_xticklabels(ks)
ax.set_ylabel("Retrieval Accuracy")
ax.set_title("IU X-Ray Image→Text Retrieval (val)")
ax.legend()
plt.tight_layout()
plt.savefig("experiments/results/chexzero_iu_retrieval.png", dpi=150)
plt.show()
print("Plot saved to experiments/results/chexzero_iu_retrieval.png")

---
## 5. Zero-Shot Evaluation on NIH ChestX-ray14

**Manual download required** — NIH-14 is not publicly accessible via `wget`.

Steps:
1. Go to https://nihcc.app.box.com/v/ChestXray-NIHCC in your browser
2. Download `images_001.tar.gz` … `images_012.tar.gz`, `Data_Entry_2017.csv`, `test_list.txt`
3. Upload them to Google Drive and mount Drive below, **or** upload directly to Colab's `/content/` folder
4. Update `NIH14_ZIPS_DIR` in the next cell and run

In [ ]:
# ── Mount Google Drive (if NIH-14 files are stored there) ────────────────────
# from google.colab import drive
# drive.mount("/content/drive")

# Path where you uploaded the NIH-14 files (zips + CSVs)
NIH14_ZIPS_DIR = "/content/drive/MyDrive/nih_chestxray14"   # ← update this

NIH14_DIR = "/content/radiology-vision-language-models/data/nih_chestxray14"
import os
os.makedirs(NIH14_DIR + "/images", exist_ok=True)
print("NIH-14 dir:", NIH14_DIR)

In [ ]:
# ── Extract NIH-14 image archives ────────────────────────────────────────────
import tarfile, glob
from pathlib import Path

images_dir = Path(NIH14_DIR) / "images"
archives   = sorted(glob.glob(f"{NIH14_ZIPS_DIR}/images_*.tar.gz"))

if not archives:
    print("No image archives found at", NIH14_ZIPS_DIR)
    print("Update NIH14_ZIPS_DIR and re-run this cell.")
else:
    for arc in archives:
        print(f"Extracting {Path(arc).name}…")
        with tarfile.open(arc, "r:gz") as tar:
            tar.extractall(images_dir)
    print(f"Total images: {len(list(images_dir.rglob('*.png')))}  (expected ~112,120)")

# Copy metadata
import shutil
for fname in ["Data_Entry_2017.csv", "test_list.txt", "train_val_list.txt"]:
    src = Path(NIH14_ZIPS_DIR) / fname
    if src.exists():
        shutil.copy(src, Path(NIH14_DIR) / fname)
        print(f"Copied {fname}")

In [ ]:
# ── Run zero-shot evaluation ─────────────────────────────────────────────────
import torch
import numpy as np
from torch.utils.data import DataLoader

from src.data_loaders.nih_chestxray14 import NIHChestXray14Dataset, NIH14_LABELS
from src.data_loaders.transforms import get_val_transforms
from src.evaluation.zero_shot import compute_zero_shot_scores
from src.evaluation.metrics import compute_auroc, compute_auprc, classification_report
from src.models.chexzero import CheXzero
from src.utils.config import load_config

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load trained model
CHECKPOINT = "experiments/results/checkpoints/chexzero/best.pt"
# Uncomment to load from Drive instead:
# CHECKPOINT = "/content/drive/MyDrive/chexzero_best.pt"

config = load_config("experiments/configs/chexzero.yaml")
model  = CheXzero(embed_dim=config["model"]["embed_dim"])
ckpt   = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device).eval()
print(f"Checkpoint loaded (epoch {ckpt.get('epoch', '?')}).")

# Dataset
test_dataset = NIHChestXray14Dataset(
    data_dir=NIH14_DIR,
    split="test",
    split_csv=f"{NIH14_DIR}/test_list.txt",
    transform=get_val_transforms(config["data"]["image_size"]),
)
print(f"Test samples: {len(test_dataset)}")

test_loader = DataLoader(
    test_dataset,
    batch_size=32,   # T4-safe; increase to 64 for A100
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

# Zero-shot scores
print("Computing zero-shot scores…")
scores_dict = compute_zero_shot_scores(
    model=model,
    image_loader=test_loader,
    labels=NIH14_LABELS,
    device=device,
)

# Ground-truth labels
all_labels = {l: [] for l in NIH14_LABELS}
for batch in test_loader:
    gt = batch["labels"].numpy()
    for i, label in enumerate(NIH14_LABELS):
        all_labels[label].append(gt[:, i])
labels_dict = {l: np.concatenate(all_labels[l]) for l in NIH14_LABELS}

# Metrics
auroc_results = compute_auroc(scores_dict, labels_dict)
auprc_results = compute_auprc(scores_dict, labels_dict)

score_matrix = np.stack([scores_dict[l] for l in NIH14_LABELS], axis=1)
label_matrix = np.stack([labels_dict[l] for l in NIH14_LABELS], axis=1)

print("\n=== Zero-Shot Classification Results (NIH ChestX-ray14) ===")
print(classification_report(score_matrix, label_matrix, label_names=NIH14_LABELS))
print(f"\nMean AUROC : {auroc_results['mean_auroc']:.4f}")
print(f"Mean AUPRC : {auprc_results['mean_auprc']:.4f}")
print("\nTarget: mean AUROC ≥ 0.75 (zero-shot, no NIH-14 training data)")

---
## 6. Results Visualisation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels_to_plot = [l for l in NIH14_LABELS if l in auroc_results]
auroc_vals     = [auroc_results[l] for l in labels_to_plot]
mean_auroc     = auroc_results["mean_auroc"]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(labels_to_plot, auroc_vals, color="steelblue")
ax.axvline(mean_auroc, color="red", linestyle="--", label=f"Mean AUROC = {mean_auroc:.3f}")
ax.axvline(0.75, color="orange", linestyle=":", label="Target (0.75)")
ax.set_xlabel("AUROC")
ax.set_title("CheXzero Zero-Shot AUROC on NIH ChestX-ray14")
ax.set_xlim(0, 1)
ax.legend()
plt.tight_layout()
plt.savefig("experiments/results/chexzero_auroc.png", dpi=150)
plt.show()
print("Plot saved to experiments/results/chexzero_auroc.png")